## Lesson contract: Retrieval strategies

**Scenario.** Compare lexical, dense-like, and hybrid evidence for an incident. This notebook is the primary lesson: the theory, design trade-offs, runnable implementation, experiment, and reflection live together. The examples are deterministic so they run without credentials; replace the adapter boundary with a hosted model or vector service only after the behavior is tested.

### Core idea

A reliable RAG system separates evidence acquisition from answer generation. It preserves source identity, metadata, and a trace of decisions. Retrieval can fail because the corpus is incomplete, stale, unauthorized, or ambiguous; therefore a production answer needs a confidence policy and an explicit abstention path.

### Architecture map

```text
FLOW (read top to bottom)

+--------------------+
| Question           |
+--------------------+
          |
          v
+--------------------+
| Retrieve evidence  |
+--------------------+
          |
          v
+--------------------+
| Filter and rank    |
+--------------------+
          |
          v
+--------------------+
| Bounded context    |
+--------------------+
          |
          v
+------------------------+
| Generate or synthesize |
+------------------------+
          |
          v
+--------------------+
| Supported?         |
+--------------------+

Decision branches:
  +-- yes --> [Cited answer]
  `-- no --> [Abstain or recover]
```

Before running code, write down which state crosses each arrow and which component is allowed to make a model decision.

In [ ]:
from dataclasses import dataclass
from collections import Counter
import re

@dataclass(frozen=True)
class Evidence:
    doc_id: str
    text: str
    source: str
    metadata: dict

CORPUS = [
    Evidence("runbook-17", "Correlate checkout errors with the 08:42 deployment before proposing rollback.", "runbooks/checkout.md", {"tenant":"acme","team":"payments","version":3}),
    Evidence("incident-22", "A European checkout timeout followed a dependency latency spike; no records were lost.", "incidents/22.md", {"tenant":"acme","team":"payments","version":5}),
    Evidence("policy-04", "Enterprise customers receive a status update within 30 minutes of a confirmed incident.", "policies/sla.md", {"tenant":"acme","team":"support","version":7}),
]

def terms(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def retrieve(query, k=2, tenant="acme"):
    q=Counter(terms(query))
    allowed=[d for d in CORPUS if d.metadata.get("tenant")==tenant]
    return sorted(allowed, key=lambda d: sum((q & Counter(terms(d.text))).values()), reverse=True)[:k]

question="Checkout is timing out in Europe after a deployment. What should support do?"
hits=retrieve(question)
[(h.doc_id,h.source,h.metadata) for h in hits]

### Example 1 — inspect and cite evidence

Do not pass opaque strings to a model. The context builder keeps stable IDs, source paths, versions, and tenant metadata so a reviewer can reproduce the answer and an authorization layer can audit inclusion. Remove one field and discuss which guarantee disappears.

In [ ]:
def build_context(hits):
    return "\n".join(f"[{h.doc_id}] {h.text} (source={h.source}; version={h.metadata['version']})" for h in hits)

context=build_context(hits)
answer="Investigate dependency latency and correlate it with the deployment before proposing rollback. If confirmed, update enterprise customers within 30 minutes."
print(context)
print("\nANSWER:", answer)

### Example 2 — insufficient evidence and failure recovery

The second query is intentionally unrelated. A safe system does not convert a plausible retrieved sentence into an answer. In a real implementation, replace this small lexical check with an evaluated relevance/groundedness grader, but keep the same explicit statuses: `answer`, `recover`, and `abstain`.

In [ ]:
def answer_question(query, k=2):
    selected=retrieve(query,k=k)
    evidence=" ".join(d.text.lower() for d in selected)
    required=("checkout" in query.lower() and "deployment" in evidence)
    if not selected or not required:
        return {"status":"abstain","reason":"evidence is missing or not specific enough","citations":[d.doc_id for d in selected]}
    return {"status":"answer","text":answer,"citations":[d.doc_id for d in selected]}

print(answer_question(question))
print(answer_question("Which planets have rings?"))

### Experiment and production checklist

Run the next cell with different `k` values. Record source IDs, context length, and whether the answer remains supported. Then add a stale document and a different tenant; prove they are filtered or trigger abstention.

Production checklist:

- filter authorization and tenant metadata before context assembly;
- keep original query, rewrites, candidates, and scores in a trace;
- cap context size, retries, tool calls, latency, and spend;
- monitor freshness, retrieval recall, citation coverage, abstention rate, and answer quality;
- retain a kill switch and a rollbackable index/prompt version.

### Practice

1. Add a fourth document about Payments and test a false positive.
2. Add `effective_at` and reject stale runbooks.
3. Replace the lexical scorer with a `Retriever` protocol and document where a dense or hybrid implementation fits.
4. Write one regression test for empty, contradictory, and unauthorized evidence.

Reference: https://qdrant.tech/documentation/concepts/hybrid-queries/

In [ ]:
for k in (1,2,3):
    selected=retrieve(question,k=k)
    print({"k":k,"ids":[d.doc_id for d in selected],"context_chars":len(build_context(selected))})

# Retrieval strategies

Retrieval is an information-retrieval problem inside a RAG system. This notebook compares an inspectable BM25 implementation with a simulated dense ranking and combines them with reciprocal-rank fusion.

## Why multiple signals?

Lexical retrieval is strong for exact identifiers and dense retrieval is often better at paraphrases. Raw scores from different systems are not directly comparable, so rank fusion is a practical baseline.

```text
FLOW (read top to bottom)

+--------------------+
| Query              |
+--------------------+

Decision branches:
  +-- next --> [BM25]
  `-- next --> [Dense adapter]

Supporting paths:
  [BM25] --feeds--> [RRF fusion]
  [Dense adapter] --feeds--> [RRF fusion]
```

In [ ]:
from examples.intermediate.retrieval_strategies import BM25, Document, reciprocal_rank_fusion

documents = [
    Document('auth', 'API keys use the Authorization header and can be rotated.'),
    Document('errors', 'Error E401 means the request is unauthorized.'),
    Document('billing', 'Invoices are available from the billing endpoint.'),
]
bm25 = BM25(documents)
lexical = bm25.search('unauthorized API request', top_k=3)
[(doc.doc_id, round(score, 3)) for doc, score in lexical]

In [ ]:
# A dense retriever would return Documents in this shape.
dense = [documents[1], documents[0], documents[2]]
hybrid = reciprocal_rank_fusion([doc for doc, _ in lexical], dense)
[(doc.doc_id, round(score, 4)) for doc, score in hybrid]

## Exercise

Add a paraphrase query and an exact error-code query. Create a small relevance label for each query, then compare lexical, dense, and hybrid top-k results. Do not choose a winner from one example; turn the examples into a golden evaluation set.